# GPT Base Model — Interactive Loading & Generation

Load the pretrained 34M-param base GPT checkpoint, run text completions, and (optionally) a lightweight eval pass.

This is a tinkering harness, not a polished demo — the model is small (34M params, ~1B tokens of pretrain).

## 1. Setup

In [1]:
import os

# Must be set before importing torch/chimera (HF cache location env var is read at import time).
os.environ["HF_HOME"] = "/mnt/ai/data/hf"

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

device: cuda
torch: 2.12.1+cu130


## 2. Config

Tinker with the checkpoint path and generation hyperparams here.

In [2]:
CKPT_PATH = "/mnt/ai/runs/llm/gpt/gpt-384-12-3-6-baseline-1B/checkpoints/gpt.ckpt"
TOKENIZER_ID = "LiquidAI/LFM2.5-230M"

# Model architecture (must match training config exactly to load the state_dict).
MODEL_KWARGS = dict(
    block_size=2048,
    n_embd=384,
    n_head=12,
    n_kv_head=3,
    n_layer=6,
    tie_embedding=True,
    mup_base_width=256,
    mup_base_std=0.02,
    mup_input_mult=1.0,
    mup_output_mult=1.0,
)

# Generation defaults (overridable per-call to generate()).
TEMPERATURE = 0.8
TOP_K = 50
MAX_NEW_TOKENS = 100

## 3. Load tokenizer + model

In [3]:
from chimera.models import GPT
from chimera.tokenizers import BPETokenizer

tok = BPETokenizer.from_pretrained(TOKENIZER_ID)

model = GPT(vocab_size=tok.vocab_size, **MODEL_KWARGS)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
# Lightning + torch.compile prefixes every key with "model._orig_mod.".
prefix = "model._orig_mod."
state_dict = {
    k[len(prefix) :]: v for k, v in ckpt["state_dict"].items() if k.startswith(prefix)
}
model.load_state_dict(state_dict)
model.to(device).eval()  # gotcha: chimera does NOT move the model to cuda automatically

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {CKPT_PATH}")
print(f"params: {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"vocab_size: {tok.vocab_size}")

loaded /mnt/ai/runs/llm/gpt/gpt-384-12-3-6-baseline-1B/checkpoints/gpt.ckpt
params: 34,030,080 (34.0M)
vocab_size: 64402


## 4. Sampling loop

`GPT.forward` has no high-level `.generate()` — a small autoregressive loop with temperature + top-k sampling, using the tokenizer's `<|endoftext|>` id as a stop condition.

In [4]:
eos_id = tok._tok.token_to_id("<|endoftext|>")


@torch.inference_mode()
def generate(
    prompt: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_k: int = TOP_K,
) -> str:
    """Greedy/top-k autoregressive sampling from a raw text prompt (no chat template)."""
    ids = tok._tok.encode(prompt, add_special_tokens=False).ids
    idx = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
        for _ in range(max_new_tokens):
            if idx.size(1) >= model.block_size:
                break
            logits = model(idx)[:, -1, :]  # (1, vocab)
            logits = logits / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
            if nxt.item() == eos_id:
                break

    return tok.decode(idx[0].tolist())


print("generate() ready")

generate() ready


## 5. Example completions

In [5]:
prompt = "The history of the Roman Empire began"
print(generate(prompt, max_new_tokens=80))

The history of the Roman Empire began in 1884. It was thought that the 1874 dynasty was a member of the Roman Empire and the earliest modern modern civilizations. It was founded in 1886 and has been replaced by the Roman Empire. The 1878 dynasty was defeated in the empire. Due to its 1878 dynasty, the Roman Empire, the Roman Empire, and the Roman Empire, it remained a strong


In [6]:
prompt = "def fibonacci(n):"
print(generate(prompt, max_new_tokens=80, temperature=0.6))

def fibonacci(n):
    # 计算数组合
    # 计算数组合
    # 计算数组合
    # 计算数组合
    # 计算数组合
    # 计算数组合
    # 计算数组合
    # 计算数组合
    # 计算数组合
    # 计算数组合


In [7]:
prompt = "The quadratic formula for solving ax^2 + bx + c = 0 is"
print(generate(prompt, max_new_tokens=80, temperature=0.6))

The quadratic formula for solving ax^2 + bx + c = 0 is equivalent to solving the quadratic equation ax^2 + bx + c = 0, where a = 1, b = 1, c = 0.

### Step 4: Solve the quadratic equation

The quadratic equation ax^2 + bx + c = 0 has roots - 1. The roots are x = ±1, a = 1, b


## 6. (Optional) lightweight eval

A quick val-loss/bits-per-byte check on a handful of batches from the pretrain mixture, plus (optionally, and much slower) the full zero-shot benchmark suite via `bench.py`. Both use `limit=`/few batches to stay fast — skip this section if you just wanted generations.

In [8]:
import math

import torch.nn.functional as F

from chimera.data import MixtureDataModule

NUM_VAL_BATCHES = 4  # keep this small; it's just a sanity check, not a full eval

dm = MixtureDataModule(
    mix_name="mix_1B", batch_size=8, seq_len=2048, pretrained_id=TOKENIZER_ID
)
dm.prepare_data()
dm.setup("fit")

losses = []
with (
    torch.inference_mode(),
    torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")),
):
    val_loader = dm.val_dataloader()
    for i, (x, y) in enumerate(val_loader):
        if i >= NUM_VAL_BATCHES:
            break
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        losses.append(loss.item())

mean_loss = sum(losses) / len(losses)
bpt = mean_loss / math.log(
    2
)  # bits per TOKEN (tokenizer-dependent, NOT comparable across tokenizers)
# bits per BYTE = bpt / (bytes per token); the cross-tokenizer-comparable metric.
BYTES_PER_TOKEN = 3.4914  # measured for the LFM2.5 tokenizer on mix_1B (see projects/llm/gpt/bpb.py cache)
bpb = bpt / BYTES_PER_TOKEN
print(
    f"val over {len(losses)} batches: loss {mean_loss:.4f} nats | bpt {bpt:.4f} bits/token | bpb {bpb:.4f} bits/byte"
)

val over 4 batches: loss 1.9078 nats | bpt 2.7523 bits/token | bpb 0.7883 bits/byte


In [9]:
# Optional, slower: full zero-shot benchmark suite (HellaSwag/PIQA/etc via lm-eval).
# Uncomment to run -- limited to a handful of examples per task so it stays quick.
RUN_FULL_BENCH = False

if RUN_FULL_BENCH:
    import sys

    sys.path.insert(0, "projects/llm/gpt")
    from bench import DEFAULT_TASKS, print_table, run_benchmarks

    results = run_benchmarks(
        model, tok, DEFAULT_TASKS, block_size=2048, device=device, limit=20
    )
    print_table(results)
else:
    print("RUN_FULL_BENCH is False -- skipping (set True to run the lm-eval suite)")

RUN_FULL_BENCH is False -- skipping (set True to run the lm-eval suite)
